# Statistical Significance Tests — Report Figures

Tests for the 5 main report figures. Non-parametric tests are used throughout because actionability scores are heavily right-skewed (83% zero).

**Tests used:**
- Kruskal-Wallis H: non-parametric equivalent of one-way ANOVA (3+ groups)
- Mann-Whitney U: non-parametric equivalent of independent t-test (2 groups)
- Chi-square: proportion differences across groups
- Effect sizes: η² (eta-squared) for Kruskal-Wallis, rank-biserial r for Mann-Whitney, Cramér's V for chi-square
- Post-hoc pairwise comparisons with Bonferroni correction where applicable

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import kruskal, mannwhitneyu, chi2_contingency, fisher_exact
from itertools import combinations

df = pd.read_csv('output/enriched.csv')
df['source_group'] = df['source_type'].apply(lambda x: 'National news' if x == 'national_news' else 'Other outlets')
df['imperative_combined'] = ((df['mean_imperative_count'] > 0) | (df['mean_verbs_imperative_count'] > 0)).astype(int)

print(f'Dataset: {len(df)} articles | languages: {df.language.value_counts().to_dict()}')

# ── helpers ──────────────────────────────────────────────────────────────────
def eta_squared_kruskal(h, k, n):
    """Eta-squared effect size for Kruskal-Wallis."""
    return (h - k + 1) / (n - k)

def rank_biserial_r(u, n1, n2):
    """Rank-biserial correlation for Mann-Whitney U."""
    return 1 - (2 * u) / (n1 * n2)

def cramers_v(chi2, n, k):
    """Cramér's V effect size for chi-square."""
    return np.sqrt(chi2 / (n * (k - 1)))

def bonferroni_mw(groups, data, col):
    """Pairwise Mann-Whitney U with Bonferroni correction."""
    pairs = list(combinations(groups, 2))
    alpha_adj = 0.05 / len(pairs)
    rows = []
    for g1, g2 in pairs:
        a = data.loc[data[col] == g1, 'actionability_percentage'].dropna()
        b = data.loc[data[col] == g2, 'actionability_percentage'].dropna()
        if len(a) < 2 or len(b) < 2:
            continue
        u, p = mannwhitneyu(a, b, alternative='two-sided')
        r = round(abs(rank_biserial_r(u, len(a), len(b))), 3)
        rows.append({'Comparison': f'{g1} vs {g2}', 'n1': len(a), 'n2': len(b),
                     'Mean 1': round(a.mean(), 2), 'Mean 2': round(b.mean(), 2),
                     'U': round(u), 'p (raw)': round(p, 4),
                     'p (Bonferroni)': round(min(p * len(pairs), 1.0), 4),
                     'Sig (adj)': '***' if p*len(pairs)<0.001 else ('**' if p*len(pairs)<0.01 else ('*' if p*len(pairs)<0.05 else 'ns')),
                     '|r|': r})
    return pd.DataFrame(rows)

---
## Figure 1 — Range of Actionability
**Key claim:** The distribution is strongly bimodal and differs significantly across languages.

In [ ]:
# Descriptive summary
s = df['actionability_percentage']
desc = pd.DataFrame({
    'Statistic': ['N articles', 'N zero (0%)', '% zero', 'N non-zero', '% non-zero',
                  'Corpus mean', 'Median (all)', 'Median (non-zero)', 'Max'],
    'Value': [len(s), (s==0).sum(), f"{(s==0).mean()*100:.1f}%",
              (s>0).sum(), f"{(s>0).mean()*100:.1f}%",
              f"{s.mean():.2f}%", f"{s.median():.1f}%",
              f"{s[s>0].median():.1f}%", f"{s.max():.1f}%"]
})
print('=== FIGURE 1: Descriptive Statistics ===')
print(desc.to_string(index=False))

# Kruskal-Wallis across languages
groups_lang = [df.loc[df['language']==l, 'actionability_percentage'].dropna() for l in ['en','es','pt']]
h, p = kruskal(*groups_lang)
n_total = sum(len(g) for g in groups_lang)
eta2 = eta_squared_kruskal(h, 3, n_total)

kw_table = pd.DataFrame({
    'Test': ['Kruskal-Wallis (EN vs ES vs PT)'],
    'H': [round(h, 3)], 'df': [2], 'p': [round(p, 4)],
    'Sig': ['***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else 'ns'))],
    'η²': [round(eta2, 3)]
})
print('\n=== Kruskal-Wallis: Actionability across languages ===')
print(kw_table.to_string(index=False))

# Pairwise
pw = bonferroni_mw(['en','es','pt'], df, 'language')
print('\n=== Pairwise Mann-Whitney U (Bonferroni corrected) ===')
print(pw.to_string(index=False))

---
## Figure 2 — PADM Component Presence by Language
**Key claim:** Component presence rates differ significantly across languages; advice-framing is uniformly rare.

In [ ]:
components = {
    'Imperative signals': 'imperative_combined',
    'Short-term urgency': 'mean_short_term_count',
    'Spatial anchors':    'mean_spatial_count',
    'Advice-framing':     'mean_advice',
}
langs = ['en', 'es', 'pt']
lang_labels = {'en': 'English', 'es': 'Spanish', 'pt': 'Portuguese'}

rows = []
for comp_label, col in components.items():
    # presence = 1 if any sentence triggered the feature
    df[f'has_{col}'] = (df[col] > 0).astype(int)
    contingency = pd.crosstab(df['language'], df[f'has_{col}'])
    # ensure both 0 and 1 columns exist
    for c in [0, 1]:
        if c not in contingency.columns:
            contingency[c] = 0
    contingency = contingency[[0, 1]].reindex(langs, fill_value=0)
    chi2, p, dof, _ = chi2_contingency(contingency)
    n = contingency.sum().sum()
    v = cramers_v(chi2, n, len(langs))
    pcts = {l: f"{(df.loc[df['language']==l, f'has_{col}'].mean()*100):.0f}%" for l in langs}
    rows.append({
        'Component': comp_label,
        'EN %': pcts['en'], 'ES %': pcts['es'], 'PT %': pcts['pt'],
        'χ²': round(chi2, 2), 'df': dof, 'p': round(p, 4),
        'Sig': '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else 'ns')),
        "Cramér's V": round(v, 3)
    })

fig2_table = pd.DataFrame(rows)
print('=== FIGURE 2: Chi-square — Component presence by language ===')
print(fig2_table.to_string(index=False))

# Pairwise for advice-framing (the rarest — most theoretically important)
print('\n=== Pairwise: Advice-framing presence (Bonferroni) ===')
pairs = list(combinations(langs, 2))
alpha_adj = 0.05 / len(pairs)
adv_rows = []
for l1, l2 in pairs:
    t = pd.crosstab(df.loc[df['language'].isin([l1,l2]),'language'],
                    df.loc[df['language'].isin([l1,l2]),'has_mean_advice'])
    for c in [0,1]:
        if c not in t.columns: t[c] = 0
    _, p_f = fisher_exact(t[[0,1]].values)
    adv_rows.append({
        'Comparison': f'{lang_labels[l1]} vs {lang_labels[l2]}',
        f'{lang_labels[l1]} %': f"{(df.loc[df['language']==l1,'has_mean_advice'].mean()*100):.0f}%",
        f'{lang_labels[l2]} %': f"{(df.loc[df['language']==l2,'has_mean_advice'].mean()*100):.0f}%",
        'p (Fisher)': round(p_f, 4),
        'p (Bonferroni)': round(min(p_f * len(pairs), 1.0), 4),
        'Sig': '***' if p_f*len(pairs)<0.001 else ('**' if p_f*len(pairs)<0.01 else ('*' if p_f*len(pairs)<0.05 else 'ns'))
    })
print(pd.DataFrame(adv_rows).to_string(index=False))

---
## Figure 3 — Source Type × Global Region (H₁ test)
**Key claim:** Global South scores higher but the gap is concentrated in national news, not a regional pattern.

In [ ]:
# 3a: Global North vs Global South
gn = df.loc[df['global_region']=='Global North', 'actionability_percentage'].dropna()
gs = df.loc[df['global_region']=='Global South', 'actionability_percentage'].dropna()
u, p = mannwhitneyu(gn, gs, alternative='two-sided')
r = abs(rank_biserial_r(u, len(gn), len(gs)))

region_table = pd.DataFrame([{
    'Comparison': 'Global North vs Global South',
    'n (North)': len(gn), 'Mean (North)': round(gn.mean(), 2),
    'n (South)': len(gs), 'Mean (South)': round(gs.mean(), 2),
    'U': round(u), 'p': round(p, 4),
    'Sig': '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else 'ns')),
    '|r|': round(r, 3)
}])
print('=== FIGURE 3a: Global North vs Global South ===')
print(region_table.to_string(index=False))

# 3b: Kruskal-Wallis across source types
source_types = df['source_type'].dropna().unique()
source_groups = [df.loc[df['source_type']==s, 'actionability_percentage'].dropna() for s in source_types if len(df[df['source_type']==s]) >= 5]
valid_types = [s for s in source_types if len(df[df['source_type']==s]) >= 5]
h_s, p_s = kruskal(*source_groups)
eta2_s = eta_squared_kruskal(h_s, len(source_groups), sum(len(g) for g in source_groups))

src_kw = pd.DataFrame([{
    'Test': 'Kruskal-Wallis across source types (n≥5)',
    'H': round(h_s, 3), 'df': len(source_groups)-1, 'p': round(p_s, 4),
    'Sig': '***' if p_s<0.001 else ('**' if p_s<0.01 else ('*' if p_s<0.05 else 'ns')),
    'η²': round(eta2_s, 3)
}])
print('\n=== FIGURE 3b: Kruskal-Wallis across source types ===')
print(src_kw.to_string(index=False))

# 3c: National news vs all other
nat = df.loc[df['source_group']=='National news', 'actionability_percentage'].dropna()
oth = df.loc[df['source_group']=='Other outlets', 'actionability_percentage'].dropna()
u2, p2 = mannwhitneyu(nat, oth, alternative='two-sided')
r2 = abs(rank_biserial_r(u2, len(nat), len(oth)))

nat_table = pd.DataFrame([{
    'Comparison': 'National news vs Other outlets',
    'n (National)': len(nat), 'Mean (National)': round(nat.mean(), 2),
    'n (Other)': len(oth), 'Mean (Other)': round(oth.mean(), 2),
    'U': round(u2), 'p': round(p2, 4),
    'Sig': '***' if p2<0.001 else ('**' if p2<0.01 else ('*' if p2<0.05 else 'ns')),
    '|r|': round(r2, 3)
}])
print('\n=== FIGURE 3c: National news vs all other outlets ===')
print(nat_table.to_string(index=False))

---
## Figure 4 — Frame × Source Type
**Key claim:** Source type (national vs other) only significantly raises actionability within accountability framing — not across other frames.

In [ ]:
# 4a: Kruskal-Wallis across frames (corpus level)
frame_order = ['impact', 'response', 'accountability', 'recovery']
frame_groups = [df.loc[df['dominant_frame']==f, 'actionability_percentage'].dropna() for f in frame_order]
h_f, p_f = kruskal(*frame_groups)
eta2_f = eta_squared_kruskal(h_f, 4, sum(len(g) for g in frame_groups))

frame_kw = pd.DataFrame([{
    'Test': 'Kruskal-Wallis across 4 frames',
    'H': round(h_f, 3), 'df': 3, 'p': round(p_f, 4),
    'Sig': '***' if p_f<0.001 else ('**' if p_f<0.01 else ('*' if p_f<0.05 else 'ns')),
    'η²': round(eta2_f, 3)
}])
print('=== FIGURE 4a: Kruskal-Wallis across frames ===')
print(frame_kw.to_string(index=False))

# 4b: Pairwise frame comparisons (Bonferroni)
print('\n=== FIGURE 4b: Pairwise frame comparisons (Bonferroni) ===')
pw_frames = bonferroni_mw(frame_order, df, 'dominant_frame')
print(pw_frames.to_string(index=False))

# 4c: Within each frame — national vs other (the key mediation test)
print('\n=== FIGURE 4c: National news vs Other outlets within each frame (mediation test) ===')
med_rows = []
for frame in frame_order:
    sub = df[df['dominant_frame'] == frame]
    nat_f = sub.loc[sub['source_group']=='National news', 'actionability_percentage'].dropna()
    oth_f = sub.loc[sub['source_group']=='Other outlets', 'actionability_percentage'].dropna()
    if len(nat_f) < 3 or len(oth_f) < 3:
        med_rows.append({'Frame': frame, 'n (National)': len(nat_f), 'Mean (National)': round(nat_f.mean(),2) if len(nat_f)>0 else np.nan,
                         'n (Other)': len(oth_f), 'Mean (Other)': round(oth_f.mean(),2), 'p': 'n/a (n<3)', 'Sig': '-', '|r|': '-'})
        continue
    u_f, p_mw = mannwhitneyu(nat_f, oth_f, alternative='two-sided')
    r_f = abs(rank_biserial_r(u_f, len(nat_f), len(oth_f)))
    med_rows.append({'Frame': frame,
                     'n (National)': len(nat_f), 'Mean (National)': round(nat_f.mean(), 2),
                     'n (Other)': len(oth_f), 'Mean (Other)': round(oth_f.mean(), 2),
                     'p': round(p_mw, 4),
                     'Sig': '***' if p_mw<0.001 else ('**' if p_mw<0.01 else ('*' if p_mw<0.05 else 'ns')),
                     '|r|': round(r_f, 3)})
print(pd.DataFrame(med_rows).to_string(index=False))

---
## Figure 5 — Cluster Profiles (Heatmap)
**Key claim:** The three clusters differ significantly in actionability; Cluster 1 is substantially more actionable than the others.

In [ ]:
# 5a: Kruskal-Wallis across clusters
cluster_ids = sorted(df['data_cluster_id'].dropna().unique())
cluster_groups = [df.loc[df['data_cluster_id']==c, 'actionability_percentage'].dropna() for c in cluster_ids]
cluster_labels = {c: f'Cluster {int(c)}' for c in cluster_ids}
h_c, p_c = kruskal(*cluster_groups)
eta2_c = eta_squared_kruskal(h_c, len(cluster_ids), sum(len(g) for g in cluster_groups))

clust_kw = pd.DataFrame([{
    'Test': f'Kruskal-Wallis across {len(cluster_ids)} clusters',
    'H': round(h_c, 3), 'df': len(cluster_ids)-1, 'p': round(p_c, 4),
    'Sig': '***' if p_c<0.001 else ('**' if p_c<0.01 else ('*' if p_c<0.05 else 'ns')),
    'η²': round(eta2_c, 3)
}])
print('=== FIGURE 5a: Kruskal-Wallis across clusters ===')
print(clust_kw.to_string(index=False))

# 5b: Descriptive per cluster
clust_desc = (df.groupby('data_cluster_id')['actionability_percentage']
               .agg(n='count', mean='mean', median='median', std='std')
               .round(3).reset_index())
clust_desc['data_cluster_id'] = clust_desc['data_cluster_id'].apply(lambda x: f'Cluster {int(x)}')
clust_desc.columns = ['Cluster', 'n', 'Mean %', 'Median %', 'SD']
print('\n=== FIGURE 5b: Descriptive statistics per cluster ===')
print(clust_desc.to_string(index=False))

# 5c: Pairwise cluster comparisons (Bonferroni)
print('\n=== FIGURE 5c: Pairwise cluster comparisons (Bonferroni) ===')
pairs_c = list(combinations(cluster_ids, 2))
clust_pw_rows = []
for c1, c2 in pairs_c:
    a = df.loc[df['data_cluster_id']==c1, 'actionability_percentage'].dropna()
    b = df.loc[df['data_cluster_id']==c2, 'actionability_percentage'].dropna()
    u_cp, p_cp = mannwhitneyu(a, b, alternative='two-sided')
    r_cp = abs(rank_biserial_r(u_cp, len(a), len(b)))
    p_adj = min(p_cp * len(pairs_c), 1.0)
    clust_pw_rows.append({
        'Comparison': f'Cluster {int(c1)} vs Cluster {int(c2)}',
        'n1': len(a), 'Mean 1': round(a.mean(), 2),
        'n2': len(b), 'Mean 2': round(b.mean(), 2),
        'U': round(u_cp), 'p (raw)': round(p_cp, 4),
        'p (Bonferroni)': round(p_adj, 4),
        'Sig': '***' if p_adj<0.001 else ('**' if p_adj<0.01 else ('*' if p_adj<0.05 else 'ns')),
        '|r|': round(r_cp, 3)
    })
print(pd.DataFrame(clust_pw_rows).to_string(index=False))

---
## Summary Table — All Key Tests
Paste-ready table of all significant results.

In [ ]:
summary = pd.DataFrame([
    # Figure 1
    {'Figure': '1', 'Test': 'Kruskal-Wallis: actionability across languages (EN/ES/PT)',
     'Statistic': f'H({2})={round(h,3)}', 'p': round(p,4),
     'Effect size': f'η²={round(eta_squared_kruskal(h,3,n_total),3)}',
     'Sig': '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else 'ns'))},

    # Figure 3
    {'Figure': '3', 'Test': 'Mann-Whitney U: Global North vs Global South',
     'Statistic': f'U={round(u)}', 'p': round(p2_region := mannwhitneyu(gn,gs,alternative="two-sided")[1],4),
     'Effect size': f'|r|={round(abs(rank_biserial_r(mannwhitneyu(gn,gs,alternative="two-sided")[0],len(gn),len(gs))),3)}',
     'Sig': '***' if mannwhitneyu(gn,gs,alternative='two-sided')[1]<0.001 else ('ns')},
    {'Figure': '3', 'Test': 'Kruskal-Wallis: actionability across source types',
     'Statistic': f'H({len(source_groups)-1})={round(h_s,3)}', 'p': round(p_s,4),
     'Effect size': f'η²={round(eta2_s,3)}',
     'Sig': '***' if p_s<0.001 else ('**' if p_s<0.01 else ('*' if p_s<0.05 else 'ns'))},
    {'Figure': '3', 'Test': 'Mann-Whitney U: National news vs Other outlets',
     'Statistic': f'U={round(u2)}', 'p': round(p2,4),
     'Effect size': f'|r|={round(r2,3)}',
     'Sig': '***' if p2<0.001 else ('**' if p2<0.01 else ('*' if p2<0.05 else 'ns'))},

    # Figure 4
    {'Figure': '4', 'Test': 'Kruskal-Wallis: actionability across 4 frames',
     'Statistic': f'H({3})={round(h_f,3)}', 'p': round(p_f,4),
     'Effect size': f'η²={round(eta2_f,3)}',
     'Sig': '***' if p_f<0.001 else ('**' if p_f<0.01 else ('*' if p_f<0.05 else 'ns'))},

    # Figure 5
    {'Figure': '5', 'Test': 'Kruskal-Wallis: actionability across 3 clusters',
     'Statistic': f'H({len(cluster_ids)-1})={round(h_c,3)}', 'p': round(p_c,4),
     'Effect size': f'η²={round(eta2_c,3)}',
     'Sig': '***' if p_c<0.001 else ('**' if p_c<0.01 else ('*' if p_c<0.05 else 'ns'))},
])

print('=== SUMMARY TABLE: All key statistical tests ===')
print(summary.to_string(index=False))
print('\nSig codes: *** p<0.001  ** p<0.01  * p<0.05  ns = not significant')
print('Effect size benchmarks: η² small=0.01, medium=0.06, large=0.14 | |r| small=0.1, medium=0.3, large=0.5')